# Marker Heatmaps Across Diffusion Pseudotime


This notebook uses one shared heatmap implementation for every figure. It first plots the existing human total-nephron trajectory using `total_scanpy_dpt`. It then reconstructs diffusion pseudotime independently within the broad PT, TAL, and DCT clusters and plots the same heatmap layout against those individual coordinates. Marker panels, trimming, binning, interpolation, smoothing, z-scoring, colors, and layout are held constant so the pseudospace coordinate is the intended difference between analyses.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy import sparse
from scipy.ndimage import gaussian_filter1d
from IPython.display import display

warnings.filterwarnings('ignore', category=FutureWarning)
plt.rcParams['figure.dpi'] = 120

PROJECT_DIR = Path.cwd().resolve()
DATA_DIR = PROJECT_DIR / 'data'
INPUT_PATH = DATA_DIR / 'pseudspace_analysis_caleb' / 'all_tubules_scanpy_dpt_caleb.h5ad'
BROAD_LABEL_PATH = DATA_DIR / 'all_tubules_scanpy_dpt_caleb.h5ad'
OUTPUT_DIR = DATA_DIR / 'heatmaps_caleb'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

adata = sc.read_h5ad(INPUT_PATH)
if 'lognorm' in adata.layers:
    adata.X = adata.layers['lognorm'].copy()

if 'broad_tubule_marker_call' not in adata.obs.columns:
    if not BROAD_LABEL_PATH.exists():
        raise KeyError(
            'broad_tubule_marker_call is missing from the canonical DPT file and '
            f'{BROAD_LABEL_PATH.relative_to(PROJECT_DIR)} does not exist. Rerun '
            '2_pseudospace_reconstruction.ipynb to save the cluster-derived broad labels.'
        )
    label_adata = sc.read_h5ad(BROAD_LABEL_PATH)
    if 'broad_tubule_marker_call' not in label_adata.obs.columns:
        raise KeyError(f'{BROAD_LABEL_PATH.relative_to(PROJECT_DIR)} is missing broad_tubule_marker_call')
    missing_labels = adata.obs_names.difference(label_adata.obs_names)
    if len(missing_labels):
        raise KeyError(f'{len(missing_labels)} tubules are missing from {BROAD_LABEL_PATH.relative_to(PROJECT_DIR)}')
    adata.obs['broad_tubule_marker_call'] = label_adata.obs.loc[
        adata.obs_names, 'broad_tubule_marker_call'
    ].astype(str).values
    print(f'Attached broad_tubule_marker_call from {BROAD_LABEL_PATH.relative_to(PROJECT_DIR)}')

BASE_OBS_COLUMNS = [
    'sample',
    'species',
    'condition',
    'broad_tubule_marker_call',
    'total_segment_marker_call',
    'total_scanpy_dpt',
]
missing_obs = [col for col in BASE_OBS_COLUMNS if col not in adata.obs.columns]
if missing_obs:
    raise KeyError(
        'data/pseudspace_analysis/all_tubules_scanpy_dpt.h5ad is missing required obs columns: '
        f'{missing_obs}. Regenerate this file from the final save cell in '
        '2_pseudospace_reconstruction.ipynb before running 3_heatmaps.ipynb.'
    )

print(f'Loaded {INPUT_PATH.name}: {adata.n_obs:,} tubules x {adata.n_vars:,} genes')
print(f'Loaded obs columns: {list(adata.obs.columns)}')
print(f'Heatmaps will be saved to: {OUTPUT_DIR}')




## Marker Panels

Defines the ordered PT, TAL, DCT, and whole-nephron marker panels and their display colors. Gene aliases allow the same panels to resolve against human- or mouse-style symbols in the shared ortholog matrix.


In [ ]:
PT_MARKER_GROUPS = {
    'PT-S1': [
        {'label': 'SLC5A2 / Slc5a2', 'candidates': ['SLC5A2', 'Slc5a2']},
        {'label': 'SLC5A12 / Slc5a12', 'candidates': ['SLC5A12', 'Slc5a12']},
    ],
    'PT-S2': [
        {'label': 'SLC22A8 / Slc22a8', 'candidates': ['SLC22A8', 'Slc22a8']},
        {'label': 'SLC22A6 / Slc22a6', 'candidates': ['SLC22A6', 'Slc22a6']},
        {'label': 'SLC13A3 / Slc13a3', 'candidates': ['SLC13A3', 'Slc13a3']},
        {'label': 'SLC22A2 / Slc22a2', 'candidates': ['SLC22A2', 'Slc22a2']},
        {'label': 'SLC34A1 / Slc34a1', 'candidates': ['SLC34A1', 'Slc34a1']},
    ],
    'PT-S3': [
        {'label': 'SLC7A13 / Slc7a13', 'candidates': ['SLC7A13', 'Slc7a13']},
        {'label': 'SLC5A8 / Slc5a8', 'candidates': ['SLC5A8', 'Slc5a8']},
        {'label': 'SLC38A3 / Slc38a3', 'candidates': ['SLC38A3', 'Slc38a3']},
        {'label': 'HAO2 / Hao2', 'candidates': ['HAO2', 'Hao2']},
        {'label': 'PCK1 / Pck1', 'candidates': ['PCK1', 'Pck1']},
        {'label': 'ACSM3 / Acsm3', 'candidates': ['ACSM3', 'Acsm3']},
    ],
}
PT_GROUP_ORDER = ['PT-S1', 'PT-S2', 'PT-S3']
PT_GROUP_COLORS = {'PT-S1': '#4C9BD3', 'PT-S2': '#5B8E55', 'PT-S3': '#D6B48A'}
PT_COLORS = PT_GROUP_COLORS

TAL_MARKER_GROUPS = {
    'mTAL / early TAL-like': [
        {'label': 'SLC12A1 / Slc12a1', 'candidates': ['SLC12A1', 'Slc12a1']},
        {'label': 'UMOD / Umod', 'candidates': ['UMOD', 'Umod']},
        {'label': 'KCNJ1 / Kcnj1', 'candidates': ['KCNJ1', 'Kcnj1']},
        {'label': 'BSND / Bsnd', 'candidates': ['BSND', 'Bsnd']},
        {'label': 'CLCNKB / Clcnkb', 'candidates': ['CLCNKB', 'Clcnkb']},
        {'label': 'ATP1A1 / Atp1a1', 'candidates': ['ATP1A1', 'Atp1a1']},
        {'label': 'ATP1B1 / Atp1b1', 'candidates': ['ATP1B1', 'Atp1b1']},
    ],
    'cTAL / late TAL-like': [
        {'label': 'CLDN10 / Cldn10', 'candidates': ['CLDN10', 'Cldn10']},
        {'label': 'CLDN16 / Cldn16', 'candidates': ['CLDN16', 'Cldn16']},
        {'label': 'CLDN19 / Cldn19', 'candidates': ['CLDN19', 'Cldn19']},
        {'label': 'CASR / Casr', 'candidates': ['CASR', 'Casr']},
        {'label': 'PTH1R / Pth1r', 'candidates': ['PTH1R', 'Pth1r']},
        {'label': 'VDR / Vdr', 'candidates': ['VDR', 'Vdr']},
    ],
    'Macula densa / TAL-DCT boundary': [
        {'label': 'NOS1 / Nos1', 'candidates': ['NOS1', 'Nos1']},
        {'label': 'PTGS2 / Ptgs2', 'candidates': ['PTGS2', 'Ptgs2']},
    ],
}
TAL_GROUP_ORDER = ['mTAL / early TAL-like', 'cTAL / late TAL-like', 'Macula densa / TAL-DCT boundary']
TAL_GROUP_COLORS = {'mTAL / early TAL-like': '#4C78A8', 'cTAL / late TAL-like': '#F58518', 'Macula densa / TAL-DCT boundary': '#E45756'}
TAL_COLORS = TAL_GROUP_COLORS

DCT_MARKER_GROUPS = {
    'DCT1': [
        {'label': 'SLC12A3 / Slc12a3', 'candidates': ['SLC12A3', 'Slc12a3']},
        {'label': 'PVALB / Pvalb', 'candidates': ['PVALB', 'Pvalb']},
    ],
    'DCT2': [
        {'label': 'CALB1 / Calb1', 'candidates': ['CALB1', 'Calb1']},
        {'label': 'HSD11B2 / Hsd11b2', 'candidates': ['HSD11B2', 'Hsd11b2']},
        {'label': 'SCNN1G / Scnn1g', 'candidates': ['SCNN1G', 'Scnn1g']},
    ],
}
DCT_GROUP_ORDER = ['DCT1', 'DCT2']
DCT_GROUP_COLORS = {'DCT1': '#4C78A8', 'DCT2': '#F58518'}
DCT_COLORS = DCT_GROUP_COLORS

TOTAL_NEPHRON_MARKERS = [
    {'label': 'Slc5a2 (PT-S1)', 'candidates': ['SLC5A2', 'Slc5a2']},
    {'label': 'Slc22a6 (PT-S2)', 'candidates': ['SLC22A6', 'Slc22a6']},
    {'label': 'Lrp2 (PT / pan-PT)', 'candidates': ['LRP2', 'Lrp2']},
    {'label': 'Slc9a3 (PT)', 'candidates': ['SLC9A3', 'Slc9a3']},
    {'label': 'Slc34a1 (PT / phosphate transport)', 'candidates': ['SLC34A1', 'Slc34a1']},
    {'label': 'Aqp1 (DTL / PT-DTL-associated, depending subset)', 'candidates': ['AQP1', 'Aqp1']},
    {'label': 'Clcnka (ATL)', 'candidates': ['CLCNKA', 'Clcnka']},
    {'label': 'Slc12a1 (TAL)', 'candidates': ['SLC12A1', 'Slc12a1']},
    {'label': 'Umod (TAL)', 'candidates': ['UMOD', 'Umod']},
    {'label': 'Pvalb (DCT1)', 'candidates': ['PVALB', 'Pvalb']},
    {'label': 'Slc12a3 (DCT)', 'candidates': ['SLC12A3', 'Slc12a3']},
    {'label': 'Calb1 (DCT2/CNT)', 'candidates': ['CALB1', 'Calb1']},
    {'label': 'Aqp2 (CCD / principal cell)', 'candidates': ['AQP2', 'Aqp2']},
    {'label': 'Scnn1g (CNT/CD principal cell)', 'candidates': ['SCNN1G', 'Scnn1g']},
    {'label': 'Slc4a1 (OMCD / intercalated cell)', 'candidates': ['SLC4A1', 'Slc4a1']},
    {'label': 'Slc26a4 (OMCD / pendrin+ intercalated cell)', 'candidates': ['SLC26A4', 'Slc26a4']},
    {'label': 'Atp6v1g3 (CD intercalated cell)', 'candidates': ['ATP6V1G3', 'Atp6v1g3']},
]
TOTAL_MARKERS = TOTAL_NEPHRON_MARKERS
print('Marker panels defined for PT, TAL, DCT, and TOTAL heatmaps.')


## Shared Heatmap Plotting Function

Defines the utilities and single plotting function used by both the total-nephron and cluster-specific analyses. Each heatmap removes the lowest and highest 5% of tubules on the displayed coordinate, bins expression into equal-width pseudospace intervals, interpolates internal empty bins, trims terminal empty bins, applies the same Gaussian smoothing, and computes per-gene z-scores. The x-axis remains proportional to the coordinate supplied to the function.


In [ ]:
def matrix_to_dense(matrix):
    if sparse.issparse(matrix):
        return matrix.toarray()
    return np.asarray(matrix)

def resolve_gene_aliases(adata_obj, marker_groups):
    var_names = pd.Index(adata_obj.var_names.astype(str))
    lookup = {name.upper(): name for name in var_names}
    alias_columns = [col for col in ['gene_symbol', 'gene_symbols', 'symbol', 'gene_name', 'feature_name', 'features'] if col in adata_obj.var.columns]
    for col in alias_columns:
        for idx_name, alias in zip(var_names, adata_obj.var[col].astype(str)):
            alias_upper = alias.upper()
            if alias_upper and alias_upper != 'NAN' and alias_upper not in lookup:
                lookup[alias_upper] = idx_name
    resolved = {}
    missing = []
    for group_name, markers in marker_groups.items():
        for marker in markers:
            match = None
            for candidate in marker['candidates']:
                match = lookup.get(candidate.upper())
                if match is not None:
                    break
            if match is None:
                missing.append(marker['label'])
            else:
                resolved[marker['label']] = match
    return resolved, missing

def resolve_marker_list(adata_obj, markers):
    resolved, missing = resolve_gene_aliases(adata_obj, {'all': markers})
    ordered = [marker['label'] for marker in markers if marker['label'] in resolved]
    return ordered, resolved, missing

print('Gene-resolution helper functions are defined.')


def marker_labels_for_group(marker_groups, group_order):
    labels = []
    for group in group_order:
        for marker in marker_groups.get(group, []):
            labels.append(marker['label'])
    return labels


def resolve_available_marker_groups(adata_obj, marker_groups, group_order):
    resolved_map, missing = resolve_gene_aliases(adata_obj, marker_groups)
    available_groups = {}
    for group in group_order:
        keep = [marker['label'] for marker in marker_groups.get(group, []) if marker['label'] in resolved_map]
        if keep:
            available_groups[group] = keep
    if not available_groups:
        preview_cols = [
            col for col in ['gene_symbol', 'gene_symbols', 'symbol', 'gene_name', 'feature_name', 'features']
            if col in adata_obj.var.columns
        ]
        raise ValueError(
            'None of the requested marker genes were found. '
            f'Checked adata.var_names and alias columns: {preview_cols if preview_cols else "none"}.'
        )
    gene_order = [gene for group in group_order if group in available_groups for gene in available_groups[group]]
    gene_idx = [adata_obj.var_names.get_loc(resolved_map[gene]) for gene in gene_order]
    return available_groups, gene_order, gene_idx, missing


def binned_interpolated_expression(adata_obj, gene_idx, pseudotime_col, n_bins=120, smooth_sigma=2.5, expression_layer='lognorm'):
    pos = adata_obj.obs[pseudotime_col].to_numpy(dtype=float)
    finite = np.isfinite(pos)
    if finite.sum() < 10:
        raise ValueError(f'Too few finite {pseudotime_col} values to build heatmap')
    adata_obj = adata_obj[finite].copy()
    pos = pos[finite]

    x_min = float(np.nanmin(pos))
    x_max = float(np.nanmax(pos))
    if np.isclose(x_min, x_max):
        x_min, x_max = x_min - 0.005, x_max + 0.005

    sort_idx = np.argsort(pos)
    pos = pos[sort_idx]
    X_obj = adata_obj.layers[expression_layer] if expression_layer in adata_obj.layers else adata_obj.X
    expr = matrix_to_dense(X_obj[:, gene_idx])[sort_idx].T

    bin_edges = np.linspace(x_min, x_max, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    bin_ids = np.digitize(pos, bin_edges) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    heatmap = np.full((len(gene_idx), n_bins), np.nan)
    occupied = np.zeros(n_bins, dtype=bool)
    for bin_i in range(n_bins):
        mask = bin_ids == bin_i
        count = int(mask.sum())
        if count > 0:
            occupied[bin_i] = True
            heatmap[:, bin_i] = expr[:, mask].mean(axis=1)

    occupied_idx = np.flatnonzero(occupied)
    if len(occupied_idx) == 0:
        raise ValueError(f'No occupied bins for {pseudotime_col}')
    first_bin = int(occupied_idx[0])
    last_bin = int(occupied_idx[-1])

    # Interpolate every internal empty bin without compressing the x coordinate.
    for row_i in range(heatmap.shape[0]):
        valid = np.isfinite(heatmap[row_i])
        valid[:first_bin] = False
        valid[last_bin + 1:] = False
        if valid.sum() == 0:
            continue
        if valid.sum() == 1:
            heatmap[row_i, first_bin:last_bin + 1] = heatmap[row_i, valid][0]
        else:
            heatmap[row_i, first_bin:last_bin + 1] = np.interp(
                bin_centers[first_bin:last_bin + 1],
                bin_centers[valid],
                heatmap[row_i, valid],
            )

    heatmap = heatmap[:, first_bin:last_bin + 1]
    bin_centers = bin_centers[first_bin:last_bin + 1]
    bin_edges = bin_edges[first_bin:last_bin + 2]

    if smooth_sigma and smooth_sigma > 0:
        heatmap = gaussian_filter1d(heatmap, sigma=smooth_sigma, axis=1, mode='nearest')

    gene_mean = np.nanmean(heatmap, axis=1, keepdims=True)
    gene_std = np.nanstd(heatmap, axis=1, keepdims=True)
    gene_std[~np.isfinite(gene_std) | (gene_std == 0)] = 1
    heatmap_z = (heatmap - gene_mean) / gene_std
    return heatmap, heatmap_z, bin_centers, bin_edges, occupied[first_bin:last_bin + 1]


def plot_marker_heatmap(
    adata_obj,
    marker_groups,
    group_order,
    group_colors,
    pseudotime_col,
    cluster_col,
    cluster_value=None,
    title=None,
    output_name='marker_heatmap.png',
    mask=None,
    target_label=None,
    strip_col=None,
    strip_order=None,
    strip_colors=None,
    n_bins=120,
    smooth_sigma=2.5,
    trim_fraction=0.05,
    z_clip=2,
    expression_layer='lognorm',
    show_module_legend=True,
    show_group_labels=True,
    figsize=(16, 10),
):
    if pseudotime_col not in adata_obj.obs.columns:
        raise KeyError(f'Missing pseudotime column: {pseudotime_col}')
    if cluster_col is not None and cluster_col not in adata_obj.obs.columns:
        raise KeyError(f'Missing cluster label column from notebook 2: {cluster_col}')

    plot_mask = np.ones(adata_obj.n_obs, dtype=bool)
    if cluster_col is not None and cluster_value is not None:
        plot_mask &= adata_obj.obs[cluster_col].astype(str).eq(str(cluster_value)).to_numpy()
    if mask is not None:
        plot_mask &= np.asarray(mask, dtype=bool)
    plot_mask &= adata_obj.obs[pseudotime_col].notna().to_numpy()
    if plot_mask.sum() == 0:
        raise ValueError(f'No tubules available for {title or output_name}')

    plot_adata = adata_obj[plot_mask].copy()
    n_cells_before_trim = int(plot_adata.n_obs)
    if not 0 <= trim_fraction < 0.5:
        raise ValueError('trim_fraction must be in [0, 0.5)')
    n_trim_per_end = int(np.floor(n_cells_before_trim * trim_fraction))
    if n_trim_per_end > 0:
        pseudotime_order = np.argsort(
            plot_adata.obs[pseudotime_col].to_numpy(dtype=float),
            kind='stable',
        )
        keep_idx = pseudotime_order[n_trim_per_end:n_cells_before_trim - n_trim_per_end]
        plot_adata = plot_adata[keep_idx].copy()
    if plot_adata.n_obs < 10:
        raise ValueError(f'Too few tubules remain after pseudospace trimming for {title or output_name}')

    available_groups, gene_order, gene_idx, missing = resolve_available_marker_groups(plot_adata, marker_groups, group_order)
    heatmap, heatmap_z, bin_centers, bin_edges, occupied_bins = binned_interpolated_expression(
        plot_adata,
        gene_idx,
        pseudotime_col,
        n_bins=n_bins,
        smooth_sigma=smooth_sigma,
        expression_layer=expression_layer,
    )
    heatmap_z = np.clip(heatmap_z, -z_clip, z_clip)

    module_names = [group for group in group_order if group in available_groups]
    gene_pos = {gene: i for i, gene in enumerate(gene_order)}
    module_scores = []
    for module_name in module_names:
        idx = [gene_pos[gene] for gene in available_groups[module_name]]
        module_scores.append(np.nanmean(heatmap_z[idx], axis=0))
    module_scores = np.vstack(module_scores)
    dominant_module = np.nanargmax(module_scores, axis=0)

    strip_values = dominant_module
    strip_cmap = ListedColormap([group_colors[name] for name in module_names])
    if strip_col is not None:
        if strip_col not in plot_adata.obs.columns:
            raise KeyError(f'Missing cluster strip column from notebook 2: {strip_col}')
        labels = plot_adata.obs[strip_col].astype(str).to_numpy()
        pos = plot_adata.obs[pseudotime_col].to_numpy(dtype=float)
        if strip_order is None:
            if hasattr(plot_adata.obs[strip_col], 'cat'):
                strip_order = [str(x) for x in plot_adata.obs[strip_col].cat.categories]
            else:
                strip_order = sorted(pd.unique(labels).astype(str))
        strip_order = [str(x) for x in strip_order if str(x) not in {'nan', 'None'}]
        strip_lookup = {label: i for i, label in enumerate(strip_order)}
        strip_values = np.full(len(bin_centers), np.nan, dtype=float)
        for bin_i in range(len(bin_centers)):
            if bin_i == len(bin_centers) - 1:
                in_bin = (pos >= bin_edges[bin_i]) & (pos <= bin_edges[bin_i + 1])
            else:
                in_bin = (pos >= bin_edges[bin_i]) & (pos < bin_edges[bin_i + 1])
            bin_labels = pd.Series(labels[in_bin]).replace({'nan': np.nan, 'None': np.nan}).dropna()
            if len(bin_labels):
                label = str(bin_labels.value_counts().idxmax())
                if label in strip_lookup:
                    strip_values[bin_i] = strip_lookup[label]
        if strip_colors is None:
            palette = plt.get_cmap('tab20').colors
            strip_colors = {label: palette[i % len(palette)] for i, label in enumerate(strip_order)}
        strip_cmap = ListedColormap([strip_colors.get(label, '#4D4D4D') for label in strip_order])

    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(3, 1, height_ratios=[1.2, 0.25, 8], hspace=0.08)
    ax_top = fig.add_subplot(gs[0])
    ax_strip = fig.add_subplot(gs[1], sharex=ax_top)
    ax_heat = fig.add_subplot(gs[2], sharex=ax_top)

    for module_idx, module_name in enumerate(module_names):
        ax_top.plot(
            bin_centers,
            module_scores[module_idx],
            color=group_colors[module_name],
            linewidth=2.5,
            label=module_name,
        )
    ax_top.set_xlim(float(bin_edges[0]), float(bin_edges[-1]))
    ax_top.set_ylabel('Module score')
    ax_top.set_xticks([])
    ax_top.spines['top'].set_visible(False)
    ax_top.spines['right'].set_visible(False)
    if show_module_legend:
        ax_top.legend(
            loc='upper right',
            frameon=False,
            ncol=min(3, max(1, len(module_names))),
            fontsize=9,
        )

    ax_strip.imshow(
        strip_values[np.newaxis, :],
        aspect='auto',
        cmap=strip_cmap,
        interpolation='nearest',
        extent=[float(bin_edges[0]), float(bin_edges[-1]), 0, 1],
    )
    ax_strip.set_yticks([])
    ax_strip.set_xticks([])
    for spine in ax_strip.spines.values():
        spine.set_visible(False)

    im = ax_heat.imshow(
        heatmap_z,
        aspect='auto',
        cmap='bwr',
        vmin=-z_clip,
        vmax=z_clip,
        interpolation='nearest',
        extent=[float(bin_edges[0]), float(bin_edges[-1]), len(gene_order) - 0.5, -0.5],
    )
    ax_heat.set_xlim(float(bin_edges[0]), float(bin_edges[-1]))
    ax_heat.set_xlabel(pseudotime_col)
    xticks = np.linspace(float(bin_edges[0]), float(bin_edges[-1]), 6)
    ax_heat.set_xticks(xticks)
    ax_heat.set_xticklabels([f'{x:.2f}' for x in xticks])
    ax_heat.set_yticks(np.arange(len(gene_order)))
    ax_heat.set_yticklabels(gene_order, fontsize=10)
    ax_heat.yaxis.tick_right()
    ax_heat.tick_params(axis='y', length=0)

    if target_label:
        ax_heat.text(
            1.01,
            1.01,
            target_label,
            transform=ax_heat.transAxes,
            ha='left',
            va='bottom',
            fontsize=10,
            bbox={'boxstyle': 'round,pad=0.3', 'facecolor': 'white', 'edgecolor': 'black', 'linewidth': 0.8},
        )

    offset = 0
    for module_name in module_names:
        genes = available_groups[module_name]
        center = offset + (len(genes) - 1) / 2
        if show_group_labels:
            ax_heat.text(
                -0.02,
                center,
                module_name,
                transform=ax_heat.get_yaxis_transform(),
                ha='right',
                va='center',
                fontsize=11,
                fontweight='bold',
                color=group_colors[module_name],
            )
        offset += len(genes)
        if offset < len(gene_order):
            ax_heat.axhline(offset - 0.5, color='black', linewidth=0.8)

    fig.subplots_adjust(left=0.10, right=0.76, top=0.92, bottom=0.08, hspace=0.08)
    cbar = fig.colorbar(im, ax=ax_heat, fraction=0.03, pad=0.12)
    cbar.set_label('Per-gene z-score')
    fig.suptitle(title or output_name, fontsize=16, fontweight='bold', y=0.98)
    fig.savefig(OUTPUT_DIR / output_name, dpi=200, bbox_inches='tight')
    plt.show()

    return {
        'n_cells': int(plot_adata.n_obs),
        'n_cells_before_trim': n_cells_before_trim,
        'n_cells_trimmed_per_end': n_trim_per_end,
        'trim_fraction_per_end': float(trim_fraction),
        'cluster_col': cluster_col,
        'cluster_value': cluster_value,
        'pseudotime_col': pseudotime_col,
        'dpt_min': float(bin_edges[0]),
        'dpt_max': float(bin_edges[-1]),
        'genes_used': gene_order,
        'missing_markers': missing,
        'module_names': module_names,
        'output_path': str((OUTPUT_DIR / output_name).relative_to(PROJECT_DIR)),
    }

print('Shared marker heatmap function is defined.')


## Heatmaps on the Global Nephron Coordinate

First plots the full human nephron marker panel using global `total_scanpy_dpt` and the `total_segment_marker_call` annotation strip from notebook 2. The following subsection then filters to PT, TAL, or DCT tubules while retaining that same global coordinate, allowing each segment's location within the whole-nephron trajectory to remain visible.


In [ ]:
total_heatmap_results = {}

human_mask = adata.obs['species'].astype(str).str.lower().str.contains('human', na=False).to_numpy()
mouse_mask = adata.obs['species'].astype(str).str.lower().str.contains('mouse', na=False).to_numpy()
mouse_control_mask = mouse_mask & adata.obs['condition'].astype(str).str.lower().eq('control').to_numpy()
mouse_aki_mask = mouse_mask & ~adata.obs['condition'].astype(str).str.lower().eq('control').to_numpy()

segment_specs = {
    'PT': {
        'markers': PT_MARKER_GROUPS,
        'order': PT_GROUP_ORDER,
        'colors': PT_GROUP_COLORS,
        'z_clip': 2,
    },
    'TAL': {
        'markers': TAL_MARKER_GROUPS,
        'order': TAL_GROUP_ORDER,
        'colors': TAL_GROUP_COLORS,
        'z_clip': 2,
    },
    'DCT': {
        'markers': DCT_MARKER_GROUPS,
        'order': DCT_GROUP_ORDER,
        'colors': DCT_GROUP_COLORS,
        'z_clip': 2,
    },
}

TOTAL_SEGMENT_COLORS = {
    'PT-S1': '#4C9BD3',
    'PT-S2': '#5B8E55',
    'PT-S3': '#D6B48A',
    'DTL': '#7E57C2',
    'ATL': '#26A69A',
    'TAL': '#F58518',
    'DCT': '#D81B60',
    'CNT': '#8D6E63',
    'CD': '#546E7A',
}
TOTAL_MARKER_GROUPS = {
    row['label']: [{'label': row['label'], 'candidates': row['candidates']}]
    for row in TOTAL_MARKERS
}
TOTAL_GROUP_ORDER = [row['label'] for row in TOTAL_MARKERS]
TOTAL_GROUP_COLORS = {group: TOTAL_SEGMENT_COLORS.get(group.split()[0], '#4D4D4D') for group in TOTAL_GROUP_ORDER}

total_heatmap_results['Human_TOTAL'] = plot_marker_heatmap(
    adata,
    TOTAL_MARKER_GROUPS,
    TOTAL_GROUP_ORDER,
    TOTAL_GROUP_COLORS,
    pseudotime_col='total_scanpy_dpt',
    cluster_col='total_segment_marker_call',
    cluster_value=None,
    mask=human_mask,
    target_label='Human only',
    strip_col='total_segment_marker_call',
    strip_colors=TOTAL_SEGMENT_COLORS,
    title='Human nephron markers across total cluster-defined Scanpy DPT',
    output_name='human_total_marker_heatmap.png',
    n_bins=120,
    smooth_sigma=2.5,
    z_clip=3,
    show_module_legend=False,
    show_group_labels=False,
    figsize=(18, 11),
)

summary = pd.DataFrame(total_heatmap_results).T
summary.index.name = 'heatmap'
summary.to_csv(OUTPUT_DIR / 'total_nephron_heatmap_summary.csv')
display(summary[['n_cells_before_trim', 'n_cells_trimmed_per_end', 'n_cells', 'cluster_col', 'cluster_value', 'pseudotime_col', 'dpt_min', 'dpt_max', 'output_path']])


### PT, TAL, and DCT on Global Pseudospace

Plots each broad tubule segment separately for human, healthy mouse, and AKI mouse while retaining `total_scanpy_dpt` as the x-coordinate. The cells are filtered to PT, TAL, or DCT, but their positions are not renormalized within the segment. The shared heatmap function applies the same endpoint trimming, binning, interpolation, smoothing, z-scoring, and layout used for all other panels.


In [ ]:
global_segment_results = {}

for segment, spec in segment_specs.items():
    global_segment_results[f'Human_{segment}'] = plot_marker_heatmap(
        adata,
        spec['markers'],
        spec['order'],
        spec['colors'],
        pseudotime_col='total_scanpy_dpt',
        cluster_col='broad_tubule_marker_call',
        cluster_value=segment,
        mask=human_mask,
        target_label='Human only',
        title=f'Human {segment} markers on global nephron Scanpy DPT',
        output_name=f'human_{segment.lower()}_global_dpt_marker_heatmap.png',
        n_bins=120,
        smooth_sigma=2.5,
        z_clip=spec['z_clip'],
    )

for condition_key, condition_label, condition_mask in [
    ('Mouse_Control', 'Mouse Control', mouse_control_mask),
    ('Mouse_AKI', 'Mouse AKI', mouse_aki_mask),
]:
    for segment, spec in segment_specs.items():
        global_segment_results[f'{condition_key}_{segment}'] = plot_marker_heatmap(
            adata,
            spec['markers'],
            spec['order'],
            spec['colors'],
            pseudotime_col='total_scanpy_dpt',
            cluster_col='broad_tubule_marker_call',
            cluster_value=segment,
            mask=condition_mask,
            target_label=condition_label,
            title=f'{condition_label} {segment} markers on global nephron Scanpy DPT',
            output_name=(
                f'{condition_key.lower()}_{segment.lower()}_'
                'global_dpt_marker_heatmap.png'
            ),
            n_bins=120,
            smooth_sigma=2.5,
            z_clip=spec['z_clip'],
        )

global_segment_summary = pd.DataFrame(global_segment_results).T
global_segment_summary.index.name = 'heatmap'
global_segment_summary.to_csv(
    OUTPUT_DIR / 'global_segment_heatmap_summary.csv',
)
display(
    global_segment_summary[
        [
            'n_cells_before_trim',
            'n_cells_trimmed_per_end',
            'n_cells',
            'cluster_value',
            'pseudotime_col',
            'dpt_min',
            'dpt_max',
            'output_path',
        ]
    ]
)


## Cluster-Specific Diffusion Pseudotime

Reconstructs diffusion pseudotime from scratch within each broad Leiden-derived PT, TAL, and DCT subset. For each cluster, the analysis rebuilds the neighbor graph from `X_harmony`, recomputes the diffusion map and DPT, chooses an early root using the ordered marker modules, orients the coordinate toward the late module, and scales finite values using the 5th and 95th percentiles.

The resulting in-memory columns are `pt_subset_scanpy_dpt`, `tal_subset_scanpy_dpt`, and `dct_subset_scanpy_dpt`. They do not overwrite the global DPT coordinate or the saved AnnData file.


In [ ]:
from scipy.stats import spearmanr


SUBSET_DPT_SPECS = {
    'PT': {
        'marker_groups': PT_MARKER_GROUPS,
        'group_order': PT_GROUP_ORDER,
        'pseudotime_col': 'pt_subset_scanpy_dpt',
    },
    'TAL': {
        'marker_groups': TAL_MARKER_GROUPS,
        'group_order': TAL_GROUP_ORDER,
        'pseudotime_col': 'tal_subset_scanpy_dpt',
    },
    'DCT': {
        'marker_groups': DCT_MARKER_GROUPS,
        'group_order': DCT_GROUP_ORDER,
        'pseudotime_col': 'dct_subset_scanpy_dpt',
    },
}


def robust_scale_rows(values):
    values = np.asarray(values, dtype=float)
    mean = np.nanmean(values, axis=1, keepdims=True)
    std = np.nanstd(values, axis=1, keepdims=True)
    std[~np.isfinite(std) | (std == 0)] = 1.0
    return (values - mean) / std


def recompute_subset_dpt(
    adata_obj,
    segment,
    marker_groups,
    group_order,
    output_col,
    n_neighbors=30,
    random_state=0,
):
    subset_mask = adata_obj.obs['broad_tubule_marker_call'].astype(str).eq(segment).to_numpy()
    if subset_mask.sum() < 30:
        raise ValueError(f'{segment}: too few tubules for subset-specific DPT')

    subset = adata_obj[subset_mask].copy()
    representation = next(
        (key for key in ['X_harmony', 'X_pca'] if key in subset.obsm),
        None,
    )
    if representation is None:
        raise KeyError(f'{segment}: neither X_harmony nor X_pca is available')

    available_groups, gene_order, gene_idx, missing = resolve_available_marker_groups(
        subset,
        marker_groups,
        group_order,
    )
    available_order = [group for group in group_order if group in available_groups]
    if len(available_order) < 2:
        raise ValueError(f'{segment}: at least two ordered marker modules are required')

    expression = subset.layers['lognorm'] if 'lognorm' in subset.layers else subset.X
    marker_expression = matrix_to_dense(expression[:, gene_idx]).T
    marker_z = robust_scale_rows(marker_expression)
    gene_position = {label: i for i, label in enumerate(gene_order)}

    early_group = available_order[0]
    late_group = available_order[-1]
    early_idx = [gene_position[label] for label in available_groups[early_group]]
    late_idx = [gene_position[label] for label in available_groups[late_group]]
    early_score = np.nanmean(marker_z[early_idx], axis=0)
    late_score = np.nanmean(marker_z[late_idx], axis=0)
    marker_axis = late_score - early_score

    neighbors_key = f'{segment.lower()}_subset_dpt_neighbors'
    n_neighbors_actual = min(int(n_neighbors), subset.n_obs - 1)
    sc.pp.neighbors(
        subset,
        n_neighbors=n_neighbors_actual,
        use_rep=representation,
        key_added=neighbors_key,
        random_state=random_state,
    )
    sc.tl.diffmap(subset, neighbors_key=neighbors_key)

    finite_axis = np.isfinite(marker_axis)
    if finite_axis.sum() < 10:
        raise ValueError(f'{segment}: too few finite marker-axis values')
    early_cutoff = np.nanquantile(marker_axis[finite_axis], 0.05)
    root_candidates = np.flatnonzero(finite_axis & (marker_axis <= early_cutoff))
    embedding = np.asarray(subset.obsm[representation], dtype=float)
    root_centroid = np.nanmedian(embedding[root_candidates], axis=0)
    root_distance = np.linalg.norm(embedding[root_candidates] - root_centroid, axis=1)
    root_index = int(root_candidates[np.nanargmin(root_distance)])

    subset.uns['iroot'] = root_index
    sc.tl.dpt(subset, neighbors_key=neighbors_key)
    raw_dpt = subset.obs['dpt_pseudotime'].to_numpy(dtype=float)
    finite = np.isfinite(raw_dpt) & np.isfinite(marker_axis)
    if finite.sum() < 10:
        raise ValueError(f'{segment}: DPT produced too few finite values')

    orientation_corr = spearmanr(raw_dpt[finite], marker_axis[finite]).correlation
    oriented = raw_dpt.copy()
    if np.isfinite(orientation_corr) and orientation_corr < 0:
        oriented = -oriented
    finite_oriented = np.isfinite(oriented)
    p5, p95 = np.nanpercentile(oriented[finite_oriented], [5, 95])
    scaled = np.full(oriented.shape, np.nan, dtype=float)
    scaled[finite_oriented] = np.clip(
        (oriented[finite_oriented] - p5) / max(p95 - p5, 1e-8),
        0,
        1,
    )

    full_values = np.full(adata_obj.n_obs, np.nan, dtype=float)
    full_values[subset_mask] = scaled
    adata_obj.obs[output_col] = full_values

    return {
        'segment': segment,
        'n_tubules': int(subset.n_obs),
        'n_finite_dpt': int(np.isfinite(scaled).sum()),
        'representation': representation,
        'n_neighbors': int(n_neighbors_actual),
        'early_module': early_group,
        'late_module': late_group,
        'root_obs_name': str(subset.obs_names[root_index]),
        'root_marker_axis': float(marker_axis[root_index]),
        'raw_dpt_marker_spearman': float(orientation_corr),
        'scaled_dpt_marker_spearman': float(
            spearmanr(scaled[np.isfinite(scaled)], marker_axis[np.isfinite(scaled)]).correlation
        ),
        'missing_markers': '; '.join(missing),
        'pseudotime_col': output_col,
    }


subset_dpt_diagnostics = []
for segment, spec in SUBSET_DPT_SPECS.items():
    subset_dpt_diagnostics.append(
        recompute_subset_dpt(
            adata,
            segment=segment,
            marker_groups=spec['marker_groups'],
            group_order=spec['group_order'],
            output_col=spec['pseudotime_col'],
        )
    )

subset_dpt_diagnostics = pd.DataFrame(subset_dpt_diagnostics)
subset_dpt_diagnostics.to_csv(
    OUTPUT_DIR / 'subset_recomputed_dpt_diagnostics.csv',
    index=False,
)
display(subset_dpt_diagnostics)


### Cluster-Specific DPT Diagnostics

Displays each newly reconstructed coordinate on the saved UMAP and compares it with the same tubules' global `total_scanpy_dpt` positions. Use these panels to check orientation, disconnected ranges, and major coordinate changes before interpreting the heatmaps.


In [ ]:
fig, axes = plt.subplots(
    len(SUBSET_DPT_SPECS),
    2,
    figsize=(12, 4 * len(SUBSET_DPT_SPECS)),
    constrained_layout=True,
)

for row_i, (segment, spec) in enumerate(SUBSET_DPT_SPECS.items()):
    subset_mask = adata.obs['broad_tubule_marker_call'].astype(str).eq(segment).to_numpy()
    umap = np.asarray(adata.obsm['X_umap'])[subset_mask]
    subset_dpt = adata.obs.loc[subset_mask, spec['pseudotime_col']].to_numpy(dtype=float)
    total_dpt = adata.obs.loc[subset_mask, 'total_scanpy_dpt'].to_numpy(dtype=float)

    axes[row_i, 0].scatter(
        np.asarray(adata.obsm['X_umap'])[:, 0],
        np.asarray(adata.obsm['X_umap'])[:, 1],
        s=1,
        color='#E0E0E0',
        alpha=0.2,
        linewidths=0,
        rasterized=True,
    )
    scatter = axes[row_i, 0].scatter(
        umap[:, 0],
        umap[:, 1],
        c=subset_dpt,
        cmap='viridis',
        vmin=0,
        vmax=1,
        s=4,
        linewidths=0,
        rasterized=True,
    )
    axes[row_i, 0].set_title(f'{segment}: subset-recomputed DPT')
    axes[row_i, 0].set_xticks([])
    axes[row_i, 0].set_yticks([])
    fig.colorbar(scatter, ax=axes[row_i, 0], label=spec['pseudotime_col'])

    finite = np.isfinite(total_dpt) & np.isfinite(subset_dpt)
    axes[row_i, 1].hexbin(
        total_dpt[finite],
        subset_dpt[finite],
        gridsize=55,
        mincnt=1,
        cmap='magma',
    )
    corr = spearmanr(total_dpt[finite], subset_dpt[finite]).correlation
    axes[row_i, 1].set_title(f'{segment}: total vs subset DPT (Spearman={corr:.2f})')
    axes[row_i, 1].set_xlabel('total_scanpy_dpt')
    axes[row_i, 1].set_ylabel(spec['pseudotime_col'])

fig.savefig(
    OUTPUT_DIR / 'subset_recomputed_dpt_diagnostics.png',
    dpi=220,
    bbox_inches='tight',
)
plt.show()


## Heatmaps on Individual Cluster Pseudospace

Plots human, healthy-mouse, and AKI-mouse PT, TAL, and DCT markers using the DPT coordinate reconstructed within the corresponding cluster. These calls reuse the exact marker specifications and shared plotting function used above; only the supplied pseudospace column differs.


In [ ]:
subset_dpt_results = {}

# Reuse the global-coordinate heatmap specification exactly. The only changed
# input is the x-coordinate generated by DPT within each segment subset.
subset_pseudotime_columns = {
    'PT': 'pt_subset_scanpy_dpt',
    'TAL': 'tal_subset_scanpy_dpt',
    'DCT': 'dct_subset_scanpy_dpt',
}
subset_segment_specs = {
    segment: {**spec, 'pseudotime_col': subset_pseudotime_columns[segment]}
    for segment, spec in segment_specs.items()
}

for segment, spec in subset_segment_specs.items():
    subset_dpt_results[f'Human_{segment}'] = plot_marker_heatmap(
        adata,
        spec['markers'],
        spec['order'],
        spec['colors'],
        pseudotime_col=spec['pseudotime_col'],
        cluster_col='broad_tubule_marker_call',
        cluster_value=segment,
        mask=human_mask,
        target_label='Human only',
        title=f'Human {segment} markers across subset-recomputed Scanpy DPT',
        output_name=f'human_{segment.lower()}_subset_recomputed_dpt_marker_heatmap.png',
        n_bins=120,
        smooth_sigma=2.5,
        z_clip=spec['z_clip'],
    )

for condition_key, condition_label, condition_mask in [
    ('Mouse_Control', 'Mouse Control', mouse_control_mask),
    ('Mouse_AKI', 'Mouse AKI', mouse_aki_mask),
]:
    for segment, spec in subset_segment_specs.items():
        subset_dpt_results[f'{condition_key}_{segment}'] = plot_marker_heatmap(
            adata,
            spec['markers'],
            spec['order'],
            spec['colors'],
            pseudotime_col=spec['pseudotime_col'],
            cluster_col='broad_tubule_marker_call',
            cluster_value=segment,
            mask=condition_mask,
            target_label=condition_label,
            title=f'{condition_label} {segment} markers across subset-recomputed Scanpy DPT',
            output_name=(
                f'{condition_key.lower()}_{segment.lower()}_'
                'subset_recomputed_dpt_marker_heatmap.png'
            ),
            n_bins=120,
            smooth_sigma=2.5,
            z_clip=spec['z_clip'],
        )

subset_dpt_summary = pd.DataFrame(subset_dpt_results).T
subset_dpt_summary.index.name = 'heatmap'
subset_dpt_summary.to_csv(
    OUTPUT_DIR / 'subset_recomputed_dpt_heatmap_summary.csv',
)
display(
    subset_dpt_summary[
        [
            'n_cells_before_trim',
            'n_cells_trimmed_per_end',
            'n_cells',
            'cluster_value',
            'pseudotime_col',
            'dpt_min',
            'dpt_max',
            'output_path',
        ]
    ]
)
